# Inverse design problem

Key equation 
$$ -u''(x) + αu(x)^3 + V(x)u(x) = f(x) $$
with $u(0) = u(1), u'(0) = u'(1)$

In [1]:
include("tools.jl")
using Plots

In [17]:
N = 100000
xx = linspace(0, 1, N)

α = 1.0

V = xx -> 1.0 .+ 0.5 .* cos.(2π .* xx)
f = xx -> 4π^2 .* sin.(2π .* xx) .+ α .* sin.(2pi .* xx).^3 .+ V(xx) .* sin.(2pi .* xx)


u_exact = sin.(2pi .*xx)
# p_true = create_fparams(N, α, V, f)
# u_target = Newton_solve_fourier(p_true, 20)


# V_guess = xx -> ones(length(xx))
# u_0  = Newton_solve_fourier(create_fparams(N, α, V_guess, f), 20)


f(xx)
# plot(xx, [u_target u_0], label = ["u_target" "U guess"])
# create_fparams(N, α, V, f)

100000-element Vector{Float64}:
  0.0
  0.0025747499125235995
  0.005149499815998719
  0.007724249701376881
  0.010298999559609605
  0.01287374938164841
  0.01544849915844482
  0.018023248880950353
  0.020597998540116534
  0.02317274812689488
  0.025747497632236903
  0.02832224704709414
  0.0308969963624181
  ⋮
 -0.03089699636241311
 -0.028322247047135894
 -0.025747497632252616
 -0.023172748126920942
 -0.020597998540116555
 -0.018023248880960727
 -0.015448499158429146
 -0.012873749381643089
 -0.010298999559614632
 -0.00772424970139226
 -0.0051494998160244495
 -0.002574749912523285

### Preconditioning

In [23]:
using Krylov

function Newton_solve_fourier(p::FourierParams, n_iter; verbose=false, tol=1e-16)
    u = zeros(p.N) # u_0
    for i in 1:n_iter
        res = F_fourier(u, p)
        L = (result, du) -> result .= -diff2_fourier(du, p.Ks) + 3 * p.α .* u .^2 .* du .+ p.V .* du
        op = LinearOperator(Float64, p.N, p.N, true, true, L)
        c = sum(3 * p.α .* u .^2 .+ p.V) / N
        P = (out, v) -> out .= real(ifft(fft(v) ./ (p.Ks.^2 .+ c)))
        P_op = LinearOperator(Float64, p.N, p.N, true, true, P)

        # step, stats = cg(op, res)
        step, precond_stats = cg(op, res, M=P_op)
        if(verbose)
            # _, stats = cg(op, res)
            # println("Iteration: ", i,",\n stats: ", stats )
            println("Iteration: ", i,",\n Preconditioning stats: ", precond_stats )
        end
        u = u - step
        if(sqrt(sum(step.^2))<tol)
            if(verbose)
                println("Took ", i, "iterations to converge")
            end
            return u
        end
    end
    return u
end

Newton_solve_fourier (generic function with 1 method)

In [24]:
u_solve = Newton_solve_fourier(create_fparams(100000, α, V, f), 20, verbose=true)

Iteration: 1,
 Preconditioning stats: SimpleStats
 niter: 3
 solved: true
 inconsistent: false
 indefinite: false
 npcCount: 0
 residuals: []
 Aresiduals: []
 κ₂(A): []
 allocation timer: 6.18μs
 timer: 41.86ms
 status: solution good enough given atol and rtol

Iteration: 2,
 Preconditioning stats: SimpleStats
 niter: 3
 solved: true
 inconsistent: false
 indefinite: false
 npcCount: 0
 residuals: []
 Aresiduals: []
 κ₂(A): []
 allocation timer: 1.74μs
 timer: 26.43ms
 status: solution good enough given atol and rtol

Iteration: 3,
 Preconditioning stats: SimpleStats
 niter: 3
 solved: true
 inconsistent: false
 indefinite: false
 npcCount: 0
 residuals: []
 Aresiduals: []
 κ₂(A): []
 allocation timer: 2.17μs
 timer: 21.20ms
 status: solution good enough given atol and rtol

Iteration: 4,
 Preconditioning stats: SimpleStats
 niter: 1
 solved: true
 inconsistent: false
 indefinite: false
 npcCount: 0
 residuals: []
 Aresiduals: []
 κ₂(A): []
 allocation timer: 2.61μs
 timer: 9.54ms
 sta

100000-element Vector{Float64}:
 -1.1017119964941479e-16
  6.28318530303125e-5
  0.00012566370581268692
  0.00018849555809897324
  0.0002513274096411345
  0.00031415926019117075
  0.00037699110950081783
  0.0004398229573222796
  0.0005026548034072955
  0.0005654866475079257
  0.0006283184893760827
  0.0006911503287637303
  0.0007539821654228674
  ⋮
 -0.0007539821654233204
 -0.0006911503287639875
 -0.0006283184893763571
 -0.0005654866475082289
 -0.000502654803407417
 -0.00043982295732241137
 -0.00037699110950110634
 -0.0003141592601913994
 -0.0002513274096415094
 -0.00018849555809932034
 -0.00012566370581300375
 -6.283185303054627e-5

In [25]:
sqrt(sum((u_solve - u_exact).^2))

1.1571175843410421e-11